# 01 — Explore

Start here. Everything loads through `src.load`.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from src.load import load_ml, load_raw, load_catalog, splits, feature_columns, real_only

df = load_ml()
print(f'{len(df):,} rows x {df.shape[1]} cols')
df.data_status.value_counts()

## Real vs synthetic — always check before reporting anything

In [ ]:
df.groupby(['platform','data_status']).agg(
    rows=('selling_price','size'),
    skus=('canonical_product_key','nunique'),
    snapshots=('snapshot_id','nunique'),
    median_price=('selling_price','median'))

## Who is cheapest?
Only meaningful where a SKU appears on all four platforms.

In [ ]:
four = df[df.platform_count_for_sku == 4]
four.groupby('platform').agg(
    cheapest_rate=('is_cheapest','mean'),
    median_gap_pct=('price_gap_vs_avg_pct','median')).round(3)

## Chronological split — never `train_test_split` on this

In [ ]:
train, val, test = splits(df, 'future_price')
feats = feature_columns(train)
print(f'train {len(train):,} | val {len(val):,} | test {len(test):,}')
print(f'{len(feats)} numeric features, identifiers and targets excluded')

## A first baseline
LightGBM on the chronological split. Compare against "tomorrow's price = today's price".

In [ ]:
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor

m = LGBMRegressor(n_estimators=400, learning_rate=0.05, verbose=-1)
m.fit(train[feats].fillna(-1), train.future_price)
pred = m.predict(test[feats].fillna(-1))

naive = test.selling_price                      # the baseline that must be beaten
print(f'model MAE  {mean_absolute_error(test.future_price, pred):.3f}')
print(f'naive MAE  {mean_absolute_error(test.future_price, naive):.3f}')
print('\nIf the model does not clearly beat naive, it has learned nothing useful.')

In [ ]:
pd.Series(m.feature_importances_, index=feats).sort_values(ascending=False).head(15)